<a href="https://colab.research.google.com/github/shreyashan-git/ds2002-fa26/blob/main/notebooks/04-json-apis/2026-10-09%20%E2%80%94%20Public%20API%20to%20Clean%20Table%20%E2%80%94%20Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Public API to Clean Table

**Lab — 2026-10-09 · Fall 2026**  

---

## Lab 07 — Public API to Clean Table

Build the weather-ingestion function you will actually reuse on the midterm. It has to pull real data, survive a flaky network, say plainly when it fell back to sample data, and hand back a table that is safe to join.

The last requirement is the one people skip. A function that returns *something* no matter what is worse than one that fails loudly, because the failure ends up in your findings instead of your console.

### Part 1 — a fetch you can trust

**TODO:** finish `get_weather`. Requirements:

- sets a timeout
- retries on 429 and 5xx with an increasing delay
- raises immediately on 400 or 404 — those are your bug, not theirs
- returns a DataFrame from the `daily` block

You wrote this shape in Wednesday's studio. Rebuild it from memory if you can.

In [1]:
import requests, pandas as pd, time

BASE = 'https://archive-api.open-meteo.com/v1/archive'
RETRY_STATUSES = {429, 500, 502, 503, 504}

def get_weather(start, end, lat=38.03, lon=-78.51, tries=3):
    """Daily max temperature and total precipitation for a date range."""
    params = {
        'latitude': lat, 'longitude': lon,
        'start_date': start, 'end_date': end,
        'daily': 'temperature_2m_max,precipitation_sum',
        'timezone': 'America/New_York',
    }

    for attempt in range(tries):
        try:
            r = requests.get(BASE, params=params, timeout=10)

            if r.status_code == 200:
                return pd.DataFrame(r.json()['daily'])

            if r.status_code not in RETRY_STATUSES:
                r.raise_for_status()

            r.raise_for_status()

        except requests.exceptions.RequestException as e:
            if isinstance(e, requests.exceptions.HTTPError):
                if r.status_code not in RETRY_STATUSES:
                    raise

            if attempt == tries - 1:
                raise

            time.sleep(2 ** attempt)

So basically, this function pulls daily maximum temperature and precipitation data from the API for a specific location and date range. It also handles network errors by retrying with increasing delays and raises an error if the request keeps failing.

### Part 2 — two home weekends

Pull weather for **2026-09-05** and **2026-10-03**. One range covering both is fine and is one request instead of two — be nice to free APIs.

If the call genuinely fails, use the fallback in Part 3, but you must record that you did.

In [2]:
used_fallback = False
try:
    weather = get_weather('2026-09-05', '2026-10-03')
    weather = weather[
        weather['time'].isin(['2026-09-05', '2026-10-03'])
    ].copy()

except requests.exceptions.RequestException as e:
    used_fallback = True
    weather = None
    print('API failed. Use sample data in Part 3.')
    print(e)

print(weather)
print('Used fallback:', used_fallback)

          time  temperature_2m_max  precipitation_sum
0   2026-09-05                29.4                2.5
28  2026-10-03                22.0               14.7
Used fallback: False


So basically, this code pulls weather data for the date range and filters it to only include September 5 and October 3. So if the API fails, it marks that fallback data is needed and prints an error message instead of using the real data.

### Part 3 — the fallback, labelled as a fallback

Sample values so the notebook still runs without a network. The important part is the `source` column: whoever reads your results can tell which rows are real.

In [3]:
FALLBACK = pd.DataFrame({
    'time': ['2026-09-05', '2026-10-03'],
    'temperature_2m_max': [30.1, 18.4],
    'precipitation_sum': [0.0, 22.5],
})

if used_fallback:
    used_fallback = True
    wx = FALLBACK.copy()
    wx['source'] = 'fallback sample'
else:
    wx = weather.copy()
    wx['source'] = 'open-meteo'

print(wx)

          time  temperature_2m_max  precipitation_sum      source
0   2026-09-05                29.4                2.5  open-meteo
28  2026-10-03                22.0               14.7  open-meteo


So basically, this code creates sample weather data to use if the API fails, so the notebook can still run. It also adds a source column to show whether the data came from Open-Meteo or the fallback sample.

### Part 4 — make it joinable

**TODO:** produce a frame with a real datetime column named `date`, plus `temp_max`, `precip_mm`, and `source`. Rename the API's column names to yours — your pipeline should not inherit somebody else's naming.

In [4]:
wx_clean = wx.rename(columns={
    'time': 'date',
    'temperature_2m_max': 'temp_max',
    'precipitation_sum': 'precip_mm'
}).copy()

wx_clean['date'] = pd.to_datetime(wx_clean['date'])

wx_clean = wx_clean[['date', 'temp_max', 'precip_mm', 'source']]

wx_clean = wx_clean.reset_index(drop=True)
wx = wx_clean.copy()

print(wx_clean)

        date  temp_max  precip_mm      source
0 2026-09-05      29.4        2.5  open-meteo
1 2026-10-03      22.0       14.7  open-meteo


So basically, this code cleans up the weather data by renaming the columns, converting the dates into datetime format, and keeping only the columns needed. It also resets the index and updates the original DataFrame so the data is ready to be merged with other datasets.

### Part 5 — validate before you use it

This is the habit that separates a pipeline from a script. State what must be true, and find out here rather than three cells later inside a join.

**TODO:** uncomment and make them pass.

In [5]:
assert list(wx.columns) == ['date', 'temp_max', 'precip_mm', 'source']
assert len(wx) == 2, f'expected 2 rows, got {len(wx)}'
assert pd.api.types.is_datetime64_any_dtype(wx['date'])
assert wx['precip_mm'].notna().all(), 'missing precipitation'
assert (wx['precip_mm'] >= 0).all(), 'negative rainfall is not a thing'
print('weather table is safe to join.')

weather table is safe to join.


### Part 6 — answer the question

**TODO:** which of the two dates was wetter, and by how much? Print a sentence a vendor could read, not just a number.

In [6]:
wetter = wx.loc[wx['precip_mm'].idxmax()]
drier = wx.loc[wx['precip_mm'].idxmin()]

difference = wetter['precip_mm'] - drier['precip_mm']

print(f"{wetter['date'].strftime('%B %d, %Y')} was wetter than "
      f"{drier['date'].strftime('%B %d, %Y')} by {difference:.1f} mm of rainfall.")

October 03, 2026 was wetter than September 05, 2026 by 12.2 mm of rainfall.


So basically, this code finds which of the two dates had the most and least rainfall and calculates the difference between them. It then prints a sentence explaining which date was wetter and by how much.

### Part 7 — cite it

**TODO:** print the endpoint, the coordinates, the date range, when you pulled it, and whether the data is real or fallback.

In [7]:
from datetime import datetime

print("Endpoint:", BASE)
print("Coordinates: 38.03, -78.51")
print("Date range: 2026-09-05 to 2026-10-03")
print("Pulled at:", datetime.now().strftime('%Y-%m-%d %H:%M:%S'))
print("Data source:", "fallback sample" if used_fallback else "open-meteo (real data)")

Endpoint: https://archive-api.open-meteo.com/v1/archive
Coordinates: 38.03, -78.51
Date range: 2026-09-05 to 2026-10-03
Pulled at: 2026-10-11 19:40:31
Data source: open-meteo (real data)


So basically, this code prints information about where the weather data came from, including the API endpoint, coordinates, date range, and current time. It also shows whether the data came from the real API or the fallback sample.

### Write-up

Suppose `get_weather` returned an empty DataFrame on failure instead of raising. Walk through what happens next: you merge it onto 200 days of sales and chart rain against revenue. What does the chart show, and why would nobody catch it?

So basically, if get_weather returned an empty DataFrame, merging it with 200 days of sales would leave the weather columns missing for every day in a left join. The chart would either be blank or show misleading results if the missing rainfall values were replaced with zeros. So, Nobody might catch it because the sales data would still have 200 rows, making the merge look successful. This is why the function should raise an error instead of silently returning empty data.

### Stretch

Pull the same two dates for a second location — Orlando is at roughly `28.54, -81.38` — and put both cities in one long-format frame with a `city` column. That is the exact shape the midterm needs when you compare stores in different places.

In [8]:
charlottesville = wx.copy()
charlottesville['city'] = 'Charlottesville'

orlando = get_weather(
    '2026-09-05', '2026-10-03',
    lat=28.54, lon=-81.38
)

orlando = orlando[
    orlando['time'].isin(['2026-09-05', '2026-10-03'])
].copy()

orlando = orlando.rename(columns={
    'time': 'date',
    'temperature_2m_max': 'temp_max',
    'precipitation_sum': 'precip_mm'
})

orlando['date'] = pd.to_datetime(orlando['date'])
orlando['source'] = 'open-meteo'
orlando['city'] = 'Orlando'

weather_both = pd.concat(
    [charlottesville, orlando],
    ignore_index=True
)

weather_both = weather_both[
    ['city', 'date', 'temp_max', 'precip_mm', 'source']
]

print(weather_both)

              city       date  temp_max  precip_mm      source
0  Charlottesville 2026-09-05      29.4        2.5  open-meteo
1  Charlottesville 2026-10-03      22.0       14.7  open-meteo
2          Orlando 2026-09-05      32.1        3.9  open-meteo
3          Orlando 2026-10-03      30.4       14.5  open-meteo
